<a href="https://colab.research.google.com/github/alexsandro-oliveira/fine-tunning-bert-department-classification/blob/main/fine_tunning_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import notebook_login

notebook_login()


In [2]:
%%capture

!pip install transformers
!pip install datasets
!pip install evaluate
!pip install --upgrade transformers accelerate datasets --quiet

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import os

In [4]:
dataset = load_dataset("json", data_files={"train": "/content/treino.jsonl", "test": "/content/teste.jsonl"})

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 500
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 100
    })
})

In [6]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

mapDict = {
    "suporte": 0,
    "venda": 1,
}

def transform_labels(example):
  # When batched=False, example['completion'] is a single string
  completion_value = example['completion']
  # Return the label as a list containing a single integer
  return {"label": [mapDict[completion_value]]}

def tokenize_function(example):
  return tokenizer(example['prompt'], padding=True, truncation=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=False) # Changed batched=True to batched=False
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments
import os

output_dir = "./bert-validator-test"
os.makedirs(output_dir, exist_ok=True)
os.makedirs("./logs", exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,

    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,


    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",   # ou "f1", "eval_loss"
    greater_is_better=True,

    logging_dir="./logs",
    logging_steps=100,
    logging_strategy="steps",

    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    seed=42,


    disable_tqdm=False,
)



In [13]:
from transformers import Trainer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)

In [15]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metric,
)

/tmp/ipython-input-716544377.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


TrainOutput(global_step=48, training_loss=0.779008706410726, metrics={'train_runtime': 517.5978, 'train_samples_per_second': 2.898, 'train_steps_per_second': 0.093, 'total_flos': 20041842366000.0, 'train_loss': 0.779008706410726, 'epoch': 3.0})

In [17]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.22928844392299652,
 'eval_accuracy': 1.0,
 'eval_runtime': 9.6173,
 'eval_samples_per_second': 10.398,
 'eval_steps_per_second': 1.352,
 'epoch': 3.0}

In [18]:
trainer.save_model()

In [19]:
trainer.push_to_hub("aleosantos/modelclass")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...or-test/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...or-test/model.safetensors:   4%|4         | 19.6MB /  438MB            

CommitInfo(commit_url='https://huggingface.co/aleosantos/bert-validator-test/commit/8e0d058be0002c8dc004b2a3180d56acf8b76247', commit_message='aleosantos/modelclass', commit_description='', oid='8e0d058be0002c8dc004b2a3180d56acf8b76247', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aleosantos/bert-validator-test', endpoint='https://huggingface.co', repo_type='model', repo_id='aleosantos/bert-validator-test'), pr_revision=None, pr_num=None)

In [20]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="aleosantos/bert-validator-test")

config.json:   0%|          | 0.00/835 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


In [22]:
pipe("Boa tarde! Estou querendo comprar o fone JBL Flip 6 preto pra dar de presente de Natal. Vocês estão com alguma promoção ou cupom de desconto ativo?")

[{'label': 'LABEL_1', 'score': 0.7851327657699585}]

In [23]:
pipe("Comprei uma cafeteira da marca X há 15 dias e ela começou a vazar água pela lateral. Já limpei tudo e continua acontecendo. Como faço pra acionar a garantia?")

[{'label': 'LABEL_0', 'score': 0.6973735690116882}]

In [24]:
pipe("Boa noite. O ar-condicionado split 12000 BTUs que instalei semana passada está fazendo um barulho estranho quando desliga, parece um estalo bem forte. Isso é normal ou preciso chamar assistência?")

[{'label': 'LABEL_0', 'score': 0.521309494972229}]

In [25]:
pipe("Oi, tudo bem? Vi no site de vocês o notebook Gamer com RTX 4070. Ainda tem em estoque? Qual o prazo de entrega pra São Paulo capital e dá pra parcelar em 12x sem juros?")

[{'label': 'LABEL_1', 'score': 0.7338675260543823}]